# Nemotron Nano Omni V3 GRPO Training with NemoRL

## Overview
This notebook trains **Nemotron Nano V3 Omni** to improve its mathematical visual reasoning ability using reinforcement learning from verifiable rewards (RLVR). The model starts from a supervised fine-tuned (SFT) checkpoint and learns to produce better step-by-step reasoning via trial-and-error rollouts evaluated against ground-truth answers — no human preference labels required.

The goal is to improve the model's ability to:
- Parse and understand a math diagram or chart from an image
- Reason through the problem step-by-step inside a `<think>` block
- Produce the final answer in `\boxed{...}` format

### Dataset: OpenGVLab/MMPR-Tiny

[MMPR-Tiny](https://huggingface.co/datasets/OpenGVLab/MMPR-Tiny) is a compact public subset of the **MMPR** (Multimodal Math Process Reward) dataset family, containing image–question–answer triples sourced from mathematical reasoning benchmarks.

**Notes**:
- Due to the distributed nature of the setup and training steps, all commands in this notebook are intended to be copied, pasted, and executed in the Slurm cluster head node, or interactive NemoRL shell environment, rather than from within a single JupyterLab environment.

- Interactive vs. batch training: in a production setting, it is more convenient to submit training jobs as Slurm batch jobs. However, setting up an interactive training environment allows you to iterate and debug faster. Once the interactive jobs run smoothly, you can submit them as batch training jobs. Hence we recommend exploring the interactive path first.

## Prerequisites

- **Compute**: 1× DGX H100 node (8× H100 80 GB) or equivalent on a Slurm cluster.

- **Storage**: A high-speed shared network file system (such as Lustre) for storing code, models, checkpoints, and other temporary assets. In this guide, we will assume that the shared storage is at `</YOUR/SHARED/NETWORK/STORAGE>` on the host system, to be mounted as `/shared` into the working Docker container, accessible from all nodes. We will also assume the following directory structure

```
</YOUR/SHARED/NETWORK/STORAGE> (on host):/shared (inside container)
|______code
|        |____RL  # NemoRL root directory
|        |____Nemotron/usage-cookbook/Nemotron-3-Nano-Omni/grpo  # Repository containing this notebook
|_______models
|        |____Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 # Omni model
|_______checkpoints # Store the trained model checkpoints
|_______HF_HOME   # HuggingFace cache directory
```

Note that each model checkpoint (including model's weights and optimizer's state) takes up to ~425Gb of storage. You should also account for the number of checkpoints you would like to keep (e.g., best k=3 checkpoints in the NemoRL training config). In addition, the base BF16 model checkpoint requires ~62Gb of storage, and another ~62Gb for the Megatron-converted checkpoint.

- **Model**: download the HuggingFace-format model with the [HF CLI tool](https://huggingface.co/docs/huggingface_hub/en/guides/cli) to the shared location on the high-speed storage. In this example, we start from the Nemotron Omni-V3 checkpoint downloadable from HuggingFace: [https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16](https://huggingface.co/nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16).

In [ ]:

mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/models
cd </YOUR/SHARED/NETWORK/STORAGE>/models
export HF_TOKEN=<YOUR_HF_TOKEN>

hf download nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
  --local-dir Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16

# Or from NGC:
# ngc registry model download-version "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"

- **Docker image**: Build the NemoRL Docker container

Reserve a H100 compute node. From here, clone and check out the [NemoRL Omni-v3 branch](https://gitlab-master.nvidia.com/jseppanen/nemo-rl/-/tree/nano-v3-omni-recipes/docs/omni-guides).

In [ ]:

mkdir -p </YOUR/SHARED/NETWORK/STORAGE>/code  # Put the correct path to your shared storage here
cd </YOUR/SHARED/NETWORK/STORAGE>/code

git clone --recursive -b nano-v3-omni https://github.com/NVIDIA-NeMo/RL # checkout the Nemotron nano-v3-omni-recipes branch, then checkout the submodules after

Next, from the NemoRL repo root directory, build the Docker container and push it to a central registry accessible to all nodes, such as NVIDIA NGC or Docker Hub. 

In [ ]:

cd </YOUR/SHARED/NETWORK/STORAGE>/code/RL
rm -rf .lock-venv

TAG="$(git rev-parse --short HEAD)"

docker buildx build \
  --target release \
  --build-arg MAX_JOBS=4 \
  --build-arg VLLM_PRECOMPILED_WHEEL_LOCATION="https://wheels.vllm.ai/27ca95b3c9e6e32ac02508e729c01865800fd036/vllm-0.14.0rc2.dev196%2Bg27ca95b3c-cp38-abi3-manylinux_2_31_x86_64.whl" \
  --build-arg SETUPTOOLS_SCM_PRETEND_VERSION_FOR_VLLM="0.14.0rc2.dev196+g27ca95b3c" \
  -f docker/Dockerfile \
  --tag "nemo-rl:nano-v3-vl-${TAG}" \
  --load .

**Note**: You can make use of the NVIDIA Cloud Registry NGC. See the NGC user's [guide](https://docs.nvidia.com/ngc/latest/ngc-user-guide.html) on how to set up your account and team, obtain the NGC API key for Docker login, and upload an image to NGC. 
Replace `tag` with `nvcr.io/<YOUR_NGC_ORG>/nemo-rl` (supply your correct NGC organization), or else use an accessible Docker hub tag.

Also, see the latest NemoRL Docker build [guide](https://github.com/NVIDIA-NeMo/RL/blob/main/docs/docker.md) for more information.

## Step 1. Prepare the training config file

This step defines the training recipe.

### H100 Config
The accompanied YAML config file [vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml](./vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml) is tuned for a **1× DGX H100 node** (8 GPUs), colocated setup:

| Component | Setting | Notes |
|---|---|---|
| Megatron TP | 8 | All 8 GPUs in one tensor-parallel group (intra-NVLink) |
| Megatron EP | 8 | 128 routed experts → 16 per GPU (intra-node alltoall) |
| Megatron PP | 1 | No pipeline parallel |
| vLLM TP | 8 | Full-node NVLink for fast generation |
| vLLM EP | 8 | Expert parallelism for vLLM inference |
| DP | 1 | Single replica; full node required for memory |

A practical workflow is to first run a short sanity pass (small `max_num_steps`) to validate cluster/config correctness, then scale up sequence length and rollout volume once stable.

**Notes**:
- Copy the config from the Nemotron cookbook into the NeMo RL examples directory before training (see the shell cell below).
- All paths inside the config file are container-internal paths (i.e. under `/shared/...`).

In [ ]:

NEMORL=/shared/code/RL
COOKBOOK_DIR=/shared/code/Nemotron/usage-cookbook/Nemotron-3-Nano-Omni/grpo

cp ${COOKBOOK_DIR}/vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml \
   ${NEMORL}/examples/configs/vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml

## Step 2. Interactive Ray training cluster setup

In a production setting, it is more convenient to submit training jobs as Slurm batch jobs. However, setting up an interactive training environment allows you to iterate and debug faster. Once the interactive jobs run smoothly, you can submit them as batch training jobs.

This part demonstrates how you can set up an interactive persistent Ray cluster for quick development. The procedure requires submitting a Slurm job, leaving the `COMMAND` variable empty.
 


In [ ]:

ROOT_DIR="</YOUR/SHARED/NETWORK/STORAGE>/code/RL" # NemoRL root directory on the shared storage
RAY_SUB="${ROOT_DIR}/ray.sub" # Path to the submission file ray.sub on the host

# path to the .yaml config file inside the container
CONFIG_PATH="/shared/code/RL/examples/configs/vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml"

export WANDB_API_KEY="YOUR_WANDB_KEY"
export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_H100_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl"
export MOUNTS="/lustre:/lustre,</YOUR/SHARED/NETWORK/STORAGE>:/shared" # Mount the Lustre and shared storage to the container

export JOB_NAME="grpo-nanov3omni-mmpr-tiny-1node"
export NUM_NODES="1"
export GPUS_PER_NODE="8"
export TIME_LIMIT="02:00:00"

export HF_HOME="/shared/HF_HOME"
export NCCL_DEBUG=INFO

sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  "${RAY_SUB}"

Upon successful submission, Slurm will print the SLURM_JOB_ID:
```
Submitted batch job 1980204
```

Once the Ray cluster is up, a script will be created to attach to the Ray head node. Run this script to launch experiments:
```
bash 1980204-attach.sh
``` 

See https://github.com/NVIDIA-NeMo/RL/blob/main/docs/cluster.md for further information. 


Now that you are on the head node, you can launch the command as follows:


In [ ]:

export HF_HOME=/shared/HF_HOME
export MMPR_CACHE_DIR="${HF_HOME}/mmpr_tiny_cache"
mkdir -p "${MMPR_CACHE_DIR}"
mkdir -p "${HF_HOME}/modules"  # Pre-create before workers start; avoids Lustre race on concurrent init_hf_modules() calls across nodes

export WANDB_API_KEY="YOUR_WANDB_KEY"

export TORCH_EXTENSIONS_DIR=/tmp/torch_extensions
export TRITON_CACHE_DIR=/tmp/triton_cache
export PYTHONDONTWRITEBYTECODE=1  # Prevent concurrent .pyc write races when 8 Ray workers import the same packages simultaneously

cd /opt/nemo-rl

NCCL_DEBUG=INFO \
NVTE_FWD_LAYERNORM_SM_MARGIN=16 \
NVTE_BWD_LAYERNORM_SM_MARGIN=16 \
NEMO_RL_LOG_GPU_MEMORY=1 \
CUDA_DEVICE_MAX_CONNECTIONS=1 \
NRL_IGNORE_VERSION_MISMATCH=true \
uv run examples/run_vlm_grpo.py \
 --config /shared/code/RL/examples/configs/vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml \
 policy.model_name=/shared/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16 \
 data.cache_dir=$MMPR_CACHE_DIR \
 checkpointing.checkpoint_dir=/shared/checkpoints/grpo-nanov3omni-mmpr-tiny-1node \
 logger.wandb_enabled=true \
 logger.wandb.name=grpo-nanov3omni-mmpr-tiny-1node

Known issues:
- The first run of the uv command might quickly fail due to a race condition on the Lustre file system. If this happens, simply re-run the command.

## Step 3. Starting a batch job

Once the interactive jobs run smoothly, you can launch production batch jobs with the following procedure from a Slurm login/head node.

In [ ]:

ROOT_DIR="</YOUR/SHARED/NETWORK/STORAGE>/code/RL" # NemoRL root directory on the shared storage
RAY_SUB="${ROOT_DIR}/ray.sub" # Path to the submission file ray.sub on the host

# path to the .yaml config file inside the container
CONFIG_PATH="/shared/code/RL/examples/configs/vlm_grpo_nanov3omni_mmpr_tiny_1node.yaml"

export WANDB_API_KEY="YOUR_WANDB_KEY"
export ACCOUNT="YOUR_SLURM_ACCOUNT"
export PARTITION="YOUR_H100_PARTITION"
export CONTAINER="nvcr.io/<YOUR_NGC_ORG>/nemo-rl"
export MOUNTS="/lustre:/lustre,</YOUR/SHARED/NETWORK/STORAGE>:/shared"  # Mount the Lustre and shared folder to the container

export JOB_NAME="grpo-nanov3omni-mmpr-tiny-1node"
export NUM_NODES="1"
export GPUS_PER_NODE="8"
export TIME_LIMIT="02:00:00"

export HF_HOME="/shared/HF_HOME"
export MMPR_CACHE_DIR="${HF_HOME}/mmpr_tiny_cache"
MODEL_PATH="/shared/models/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16"
CKPT_DIR="/shared/checkpoints/grpo-nanov3omni-mmpr-tiny-1node"

export NCCL_DEBUG=INFO
export TORCH_EXTENSIONS_DIR=/tmp/torch_extensions
export TRITON_CACHE_DIR=/tmp/triton_cache
export PYTHONDONTWRITEBYTECODE=1  # Prevent concurrent .pyc write races when 8 Ray workers import the same packages simultaneously

# Note: MMPR_CACHE_DIR and HF_HOME/modules are created inside the COMMAND on first run
export COMMAND="\
mkdir -p ${MMPR_CACHE_DIR} && \
mkdir -p ${HF_HOME}/modules && \
cd /opt/nemo-rl && \
NVTE_FWD_LAYERNORM_SM_MARGIN=16 \
NVTE_BWD_LAYERNORM_SM_MARGIN=16 \
NEMO_RL_LOG_GPU_MEMORY=1 \
CUDA_DEVICE_MAX_CONNECTIONS=1 \
NRL_IGNORE_VERSION_MISMATCH=true \
uv run examples/run_vlm_grpo.py \\
  --config ${CONFIG_PATH} \\
  policy.model_name=${MODEL_PATH} \\
  data.cache_dir=${MMPR_CACHE_DIR} \\
  checkpointing.checkpoint_dir=${CKPT_DIR} \\
  logger.wandb_enabled=true \\
  logger.wandb.name=${JOB_NAME}"

sbatch \
  --nodes="${NUM_NODES}" \
  --account="${ACCOUNT}" \
  --job-name="${JOB_NAME}" \
  --partition="${PARTITION}" \
  --time="${TIME_LIMIT}" \
  --gres="gpu:${GPUS_PER_NODE}" \
  --exclusive \
  --dependency=singleton \
  "${RAY_SUB}"


One the job is submitted, you can monitor the job progress with:

```
tail -f slurm-<jobID>.out
```

This prints out the initial Slurm job progression on the cluster. Once the Ray cluster a initialized and the actual job runs, you can monitor the training progress with:

```
tail -f <jobID>/ray-driver.log
```

Alternatively, if you have provided a WanDB key, you can monitor the job progress in WandDB.
